# Week 10, Part 3: The Analytics Pipeline
**Phase 2: Python for Data Analysis**

Parts 1 and 2 gave you the two halves: **get data from APIs** and **store it in SQL**. This notebook joins them into something you can run every morning without thinking: a **pipeline** that is **idempotent**, **incremental** and **observable**, then uses the resulting database to answer real questions.

This is a miniature of what a Blockchain Data Engineer does all day, and it is the engine of the Phase 2 capstone (`defi-dashboard`).

**Everything runs offline** on the fixtures. `python3 pipeline.py --live` runs the same code on real APIs (see `lesson.ipynb`, section 13).

## 0. Setup

In [ ]:
import datetime as dt
import json
import logging
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, "../../phase-1-python-fundamentals/week-05-functions-modules")
from blockchain_utils import fmt_usd, fmt_large

from api_clients import ApiError, tx_dataframe
from db_tools import connect, init_db, table_counts, latest_date
from fixtures import FixtureSession
from pipeline import run_pipeline, days_to_fetch, quality_checks

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
os.makedirs("charts", exist_ok=True)
AS_OF = "2026-10-01"          # the fixtures' "today", so results never change

## 1. ETL thinking

**ETL** = **E**xtract (get it) → **T**ransform (clean and reshape it) → **L**oad (store it). A close cousin, **ELT**, loads the raw data first and transforms inside the database, which is the Dune/dbt style you already know. Our pipeline is classic ETL because the transforms are small.

```
 EXTRACT                      TRANSFORM                    LOAD
 (network, impure)            (pure functions)             (database, side effects)
 CoinGecko  --market_chart--> parallel arrays -> DataFrame --> upsert_prices
 DefiLlama  --protocol/slug-> nested JSON    -> DataFrame --> upsert_protocol_tvl
 Etherscan  --txlist--------> string values  -> typed DF  --> upsert_transactions
```

### 1.1 Three properties that separate a script from a pipeline

| Property | Question it answers | How we get it |
|---|---|---|
| **Idempotent** | "If this runs twice (retry, cron overlap, my mistake), is the result still right?" | Natural primary keys + upsert / `INSERT OR IGNORE` |
| **Incremental** | "Does a daily run re-download 3 years of history?" | A **watermark**: the newest date already stored |
| **Observable** | "At 3 a.m., how do I know it worked, and why if it didn't?" | Logging, a run-log table, quality checks |

### 1.2 Data layers

Real warehouses organise data in layers. You'll meet these names constantly (they come from the "medallion" pattern):

| Layer | Contents | In our project |
|---|---|---|
| **Bronze / raw** | Exact API responses, untouched | `raw/` JSON snapshots (section 5) |
| **Silver / clean** | Typed, de-duplicated, constrained tables | `prices_daily`, `protocol_tvl`, `transactions` |
| **Gold / analytics** | Questions answered: metrics, joins, views | the queries in section 9 |

## 2. Separate the impure from the pure

The best habit for testable pipelines: keep **side-effecting** code (network, database) thin, and put the **logic** in **pure functions**: data in, data out, no network, no database. Pure functions can be tested with a hand-written example in two lines.

`tx_dataframe` is one: Etherscan's string-typed rows in, a typed DataFrame out. Test it with a row you craft yourself, including the nasty case, a value above int64:

In [ ]:
row = {"hash": "0xabc", "blockNumber": "20000001", "timeStamp": "1789000000", "from": "0xAAA", "to": "0xBBB",
       "value": "12500000000000000000", "gasUsed": "21000", "gasPrice": "30000000000", "isError": "0"}

out = tx_dataframe([row])
assert out.loc[0, "block_number"] == 20_000_001                    # string -> int
assert out.loc[0, "value_wei"] == "12500000000000000000"           # 12.5 ETH stays an exact TEXT string
assert out.loc[0, "gas_used"] == 21_000
print("tx_dataframe handles a >int64 value correctly")
out

No network, no database, no fixtures needed, and it runs in milliseconds. That's what you get from pure functions. (`test_week10.py`-style checks like these are what real teams run on every commit.)

## 3. The pipeline, stage by stage

`pipeline.run_pipeline` does three loads in order. Read its structure (shown here in outline; the full file is `pipeline.py`):

```python
# 1) prices: incremental per coin
for coin in coins:
    window = days_to_fetch(latest_date(conn, coin), as_of, max_days=days)   # <- the watermark
    upsert_prices(conn, cg.market_chart(coin, window))

# 2) protocols (dimension) THEN their TVL history (fact; the foreign key requires the parent first)
upsert_protocols(conn, dl.protocols())
upsert_protocol_tvl(conn, dl.protocol_tvl(slug))

# 3) transactions: immutable, so INSERT OR IGNORE
upsert_transactions(conn, tx_dataframe(list(es.iter_txlist_all(address))))
```

Notice the **load order**: parents before children, because `protocol_tvl.slug` is a foreign key to `protocols.slug`. Load them backwards and the database refuses, which is the constraint doing its job.

Run it once, then **again**, and compare:

In [ ]:
conn = init_db(connect())
s = FixtureSession()

first  = run_pipeline(conn, s, as_of=AS_OF)
counts1 = table_counts(conn)
second = run_pipeline(conn, s, as_of=AS_OF)
counts2 = table_counts(conn)

print("run 1 loaded :", first)
print("run 2 loaded :", second, "  <- far fewer prices: incremental (section 4)")
print()
for t in ["prices_daily", "protocols", "protocol_tvl", "transactions"]:
    print(f"{t:<14} {counts1[t]:>4} -> {counts2[t]:>4}   {'unchanged ✓' if counts1[t] == counts2[t] else 'CHANGED ✗'}")

**Idempotent:** four tables, identical after the second run. (`etl_runs` *is* meant to grow: it's the log, one row per load.)

## 4. Incremental loads with a watermark

Re-downloading 90 days of ETH prices every morning to get *one new day* wastes your rate-limit budget and your time. The **watermark** pattern: ask the database "what's the newest day I have?", then request only the gap.

`days_to_fetch` encodes the policy. We add a **2-day overlap** deliberately: the latest day's price is still moving and may be revised, so we re-fetch and upsert it, which is exactly what upsert is for.

In [ ]:
cases = [(None, "2026-10-01", "no data yet -> full history"),
         ("2026-09-30", "2026-10-01", "1 day behind"),
         ("2026-10-01", "2026-10-01", "up to date"),
         ("2026-09-01", "2026-10-01", "a month behind (laptop was off)"),
         ("2025-01-01", "2026-10-01", "ancient -> capped at max_days")]
for last, today, why in cases:
    print(f"last={str(last):<11} as_of={today} -> fetch {days_to_fetch(last, today):>3} days   ({why})")

# Pure function => test it like one
assert days_to_fetch(None, AS_OF) == 90
assert days_to_fetch(AS_OF, AS_OF) == 2
assert days_to_fetch("2025-01-01", AS_OF, max_days=90) == 90

In [ ]:
# Prove it against the API layer: what 'days' did each run actually request?
s = FixtureSession()
c2 = init_db(connect())
run_pipeline(c2, s, coins=["ethereum"], slugs=["lido"], as_of=AS_OF)
asked_1 = [p["days"] for url, p in s.calls if "market_chart" in url]
s.calls.clear()
run_pipeline(c2, s, coins=["ethereum"], slugs=["lido"], as_of=AS_OF)
asked_2 = [p["days"] for url, p in s.calls if "market_chart" in url]
print("first run asked for  days =", asked_1)
print("second run asked for days =", asked_2, "  <- the watermark at work")

This matters most for **Etherscan**: a wallet history can be millions of rows, so the real version of this pattern is `startblock = MAX(block_number) + 1` from your own table (a block-number watermark) instead of "days".

## 5. Keep the raw data (bronze layer)

Transforms have bugs. If you only keep the cleaned table and discover a parsing mistake in month 3, you can't redo it without re-downloading, and APIs often won't give you old data back. So **save the raw response first**, then transform. Re-running a fixed transform on saved raw data is free.

In [ ]:
def save_raw(source, name, payload, root="raw", now=None):
    '''Write one raw API response to raw/<source>/<name>_<UTC timestamp>.json and return the path.'''
    now = now or dt.datetime.now(dt.timezone.utc)
    folder = pathlib.Path(root) / source
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{name}_{now:%Y%m%dT%H%M%SZ}.json"
    path.write_text(json.dumps(payload))
    return path

with tempfile.TemporaryDirectory() as tmp:
    raw = FixtureSession().get("https://api.llama.fi/protocol/lido").json()
    p = save_raw("defillama", "protocol_lido", raw, root=tmp, now=dt.datetime(2026, 10, 1, 6, 0, tzinfo=dt.timezone.utc))
    print("saved:", p.relative_to(tmp), f"({p.stat().st_size / 1024:.0f} KB)")
    reparsed = pd.DataFrame(json.loads(p.read_text())["tvl"])        # re-transform later, no network needed
    print("re-parsed rows from disk:", len(reparsed))

In production this folder becomes object storage (S3/GCS) with Parquet files, which is Phase 5. Put `raw/` in `.gitignore`: it's data, not code.

## 6. Logging: your pipeline's black box recorder

`print` is for notebooks. A pipeline that runs unattended needs **logging**: timestamped, levelled, redirectable to a file, and filterable.

| Level | Use for |
|---|---|
| `DEBUG` | Detail for developers (each request URL, redacted) |
| `INFO` | Normal milestones ("loaded 182 price rows") |
| `WARNING` | Something odd but survivable (a quality issue) |
| `ERROR` | A step failed; the run continues or aborts |

Each module gets its own logger by name (`logging.getLogger("pipeline")`); the **application** decides where messages go with `basicConfig`. (In Jupyter, `force=True` replaces handlers the notebook already set.)

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s", stream=sys.stdout, force=True)
log_demo = init_db(connect())
_ = run_pipeline(log_demo, FixtureSession(), coins=["ethereum"], slugs=["lido"], as_of=AS_OF)

## 7. Failure isolation: one broken source must not kill the run

Sources fail independently. If Etherscan rejects your key, you still want yesterday's prices and TVL loaded. `run_pipeline` wraps each source in its own `try/except`, **logs the error loudly**, records `0` rows, and carries on. (This is one of the few places a broad `except Exception` is right: the boundary of an independent unit of work, where you log and continue. Everywhere else, catch specific exceptions.)

Simulate a bad Etherscan key:

In [ ]:
broken = init_db(connect())
summary = run_pipeline(broken, FixtureSession(), etherscan_key="", as_of=AS_OF)
print()
print("summary:", summary, "  <- prices and TVL loaded, transactions = 0")
print("quality:", quality_checks(broken, as_of=AS_OF))

Two layers of defence worked together: the **error log line** told us *why*, and the **quality check** told us *that the result is incomplete*. A pipeline that "succeeds" with an empty table is the most dangerous kind.

## 8. Data-quality checks

A data pipeline's real product is **trust**. Cheap automated checks catch the problems that otherwise surface as a wrong number in someone's presentation. `quality_checks` returns a list of human-readable problems, where empty means healthy. Checks that earn their place:

| Check | Catches |
|---|---|
| Tables non-empty | Silent source failure (just saw it) |
| No null / zero / negative prices | Parse bugs, bad API days |
| Foreign-key integrity | Orphaned rows |
| **Freshness** (`MAX(date)` vs today) | The cron job stopped three days ago |
| Row counts vs expectation | Truncated pages |

Break a database on purpose and watch the checks fire:

In [ ]:
qa = init_db(connect())
run_pipeline(qa, FixtureSession(), as_of=AS_OF)
print("healthy        :", quality_checks(qa, as_of=AS_OF))

with qa:
    qa.execute("UPDATE prices_daily SET price_usd = 0 WHERE coin_id = 'bitcoin' AND date = '2026-09-30'")
print("after bad price:", quality_checks(qa, as_of=AS_OF))

print("10 days later  :", quality_checks(qa, as_of="2026-10-11"))

**What to do with a failed check** is a design decision: in a notebook, print it; in a scheduled job, **exit non-zero** so the scheduler marks the run failed and alerts you (section 10). Never swallow it.

## 9. The payoff: analytics on your own warehouse

Everything so far was plumbing. Now ask questions. Each one is **SQL to cut the data down, Pandas to analyse**: the division of labour from Part 2.

### 9.1 How volatile is ETH?

Daily log returns and 30-day rolling volatility (annualised with √365: crypto trades every day):

In [ ]:
logging.getLogger().setLevel(logging.WARNING)        # quiet the INFO logs from here on
wh = init_db(connect())
_ = run_pipeline(wh, FixtureSession(), as_of=AS_OF)

eth = pd.read_sql_query("SELECT date, price_usd FROM prices_daily WHERE coin_id = ? ORDER BY date",
                        wh, params=("ethereum",), parse_dates=["date"]).set_index("date")
eth["log_ret"] = np.log(eth["price_usd"]).diff()
eth["vol_30d"] = eth["log_ret"].rolling(30).std() * np.sqrt(365)
print(f"latest 30-day annualised volatility: {eth['vol_30d'].iloc[-1]:.1%}")
eth.tail(3)

### 9.2 Does Lido's TVL follow the ETH price?

A join in SQL (two sources, one date key), then correlation in Pandas. Correlate **daily changes**, not levels: two trending series always look correlated, a classic spurious-correlation trap.

In [ ]:
lido = pd.read_sql_query('''SELECT t.date, t.tvl_usd, p.price_usd
                           FROM protocol_tvl t
                           JOIN prices_daily p ON p.coin_id = 'ethereum' AND p.date = t.date
                           WHERE t.slug = 'lido' ORDER BY t.date''', wh, parse_dates=["date"]).set_index("date")

print("overlapping days:", len(lido))
print("correlation of LEVELS         :", round(lido["tvl_usd"].corr(lido["price_usd"]), 3), " <- unreliable")
print("correlation of DAILY CHANGES  :", round(lido.pct_change().corr().loc["tvl_usd", "price_usd"], 3))

Our fixture series are generated **independently**, so the honest answer is "about zero", and the daily-changes line says so. Look at the **levels** line: it shows a correlation of about 0.7 between two series that have *nothing to do with each other*. That is the spurious-correlation trap in action: two random walks that both drift will "correlate" by accident. This is exactly why you correlate changes, not levels. On real data, ETH-denominated protocols like Lido show a strong positive relationship because the TVL is *priced in USD from ETH*. **Seeing the contrast between simulated and real data is itself instructive.** Build the analysis once, and it works on whichever data you point it at.

### 9.3 What do failed transactions cost?

A real insight from on-chain data: a failed transaction still **burns gas**. How much money did our demo wallet waste?

In [ ]:
pd.read_sql_query('''SELECT CASE t.is_error WHEN 0 THEN 'succeeded' ELSE 'FAILED' END AS outcome,
                           COUNT(*) AS txs,
                           ROUND(SUM(t.gas_used * CAST(t.gas_price_wei AS REAL) / 1e18 * p.price_usd), 2) AS gas_usd,
                           ROUND(AVG(t.gas_used * CAST(t.gas_price_wei AS REAL) / 1e18 * p.price_usd), 2) AS avg_gas_usd
                    FROM transactions t
                    JOIN prices_daily p ON p.coin_id = 'ethereum' AND p.date = date(t.ts, 'unixepoch')
                    GROUP BY t.is_error ORDER BY t.is_error''', wh)

### 9.4 A chart: price and TVL together (Week 9 meets Week 10)

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6), gridspec_kw={"height_ratios": [3, 2], "hspace": 0.08})
a1.plot(lido.index, lido["price_usd"], color="#3465a4", linewidth=1.8)
a1.set_ylabel("ETH price (USD)")
a1.set_title("ETH price vs Lido TVL: pulled from 2 APIs, joined in SQLite", loc="left", fontweight="bold")
a2.fill_between(lido.index, lido["tvl_usd"] / 1e9, color="#00a3ff", alpha=0.6)
a2.set_ylabel("Lido TVL ($bn)")
fig.savefig("charts/lesson10_eth_vs_lido.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Run it from the command line

A notebook is for exploring. A **pipeline** should be runnable as a command, so cron, GitHub Actions or a teammate can start it with no Jupyter. `pipeline.py` ends with the standard entry-point pattern:

```python
if __name__ == "__main__":
    sys.exit(main())
```

- `if __name__ == "__main__":` runs `main()` only when the file is executed directly, not when it's imported (that's why notebooks can `from pipeline import run_pipeline` safely)
- `argparse` gives `--live`, `--db`, `--days` flags and automatic `--help`
- `sys.exit(1)` when quality checks fail gives the scheduler a **non-zero exit code**, the universal "this failed" signal

Let's run it as a subprocess, exactly as cron would:

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    db_path = os.path.join(tmp, "chainlens.db")
    r = subprocess.run([sys.executable, "pipeline.py", "--db", db_path], capture_output=True, text=True)
    print("exit code:", r.returncode, "(0 = healthy)")
    print(r.stdout.strip())
    print("db file created:", os.path.exists(db_path), f"({os.path.getsize(db_path)/1024:.0f} KB)")

### 10.1 Scheduling (concept)

Once it runs from the command line, scheduling is just "run this command at these times":

```bash
# cron (Linux/Mac): every day at 06:00 UTC
0 6 * * *  cd ~/python-for-blockchain-analytics/phase-2-python-for-data/week-10-apis-sql && /usr/bin/python3 pipeline.py --live >> pipeline.log 2>&1
```

```yaml
# GitHub Actions: free scheduled runs (secrets live in repo settings, never in the file)
on:
  schedule: [{ cron: "0 6 * * *" }]
jobs:
  etl:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install pandas requests
      - run: python3 pipeline.py --live
        working-directory: phase-2-python-for-data/week-10-apis-sql
        env:
          ETHERSCAN_API_KEY: ${{ secrets.ETHERSCAN_API_KEY }}
```

(An Actions runner starts empty each time, so the database would need to be stored as an artifact or in a real database. That's one of the reasons Phase 5 moves to proper warehouses and orchestrators like Airflow.)

## 11. Where this goes: the Phase 2 capstone

```
              ┌── CoinGecko  ──┐
  APIs  ──────┼── DefiLlama  ──┼──►  pipeline.py  ──►  chainlens.db  ──►  SQL + Pandas  ──►  Plotly dashboard
              └── Etherscan  ──┘     (idempotent,        (SQLite,           (analysis)         (HTML you can
                                      incremental,        constraints,                          publish)
                                      logged, checked)    indexes)
```

You now own every box except the last one, which is Week 9's Plotly. The capstone is: **extend this pipeline with your own protocols and wallets, add three analyses you care about, and publish the dashboard** as a repo others can run (`git clone`, `pip install`, `python3 pipeline.py`, open `dashboard.html`).

Ideas to make it yours: Aave or Uniswap pools from DefiLlama; gas-price tracking from Etherscan's gas oracle on a schedule; a wallet watchlist; an alert when a quality check fails; a README with a screenshot of the finished dashboard.

## 12. Common pitfalls

| Pitfall | Symptom | Fix |
|---|---|---|
| Not idempotent | Duplicates after a retry | Natural primary keys + upsert |
| Re-fetching everything daily | Burned quota, slow runs | Watermark + small overlap |
| Loading children before parents | FK errors | Parents first |
| Transforming without keeping raw | Can't fix a parser bug later | Save raw responses |
| `print` instead of logging | No history when it fails at night | `logging` to a file |
| One source's failure aborts everything | Missing data from healthy sources | Isolate per source, log, continue |
| "Success" with empty tables | Wrong dashboard, nobody notices | Quality checks + non-zero exit |
| Correlating price levels | Spurious "relationships" | Correlate returns/changes |
| Secrets in the repo | Key leak | `.env`, `.gitignore`, `secrets.*` |
| Committing `*.db`, `raw/`, `.cache/` | Huge repo, stale data | `.gitignore` them |

## Lesson summary

| Concept | Idea |
|---|---|
| ETL | Extract (impure), Transform (pure), Load (side effects) |
| Idempotency | Natural keys + upsert: safe to re-run |
| Incremental | Watermark (`MAX(date)`) + short overlap |
| Layers | raw/bronze → clean/silver → analytics/gold |
| Pure functions | Easy to test with hand-made inputs |
| Logging | Levels, named loggers, `basicConfig` in the application |
| Failure isolation | Per-source `try/except`, log, continue |
| Quality checks | Non-empty, valid, referentially sound, **fresh** |
| CLI | `argparse`, `if __name__ == "__main__"`, exit codes |
| Analytics | SQL to cut down, Pandas to analyse, chart to communicate |

> 📺 **Watch / 📖 Read:** see `RESOURCES.md` for the curated list. Two good follow-ups: the free *Python for Everybody* chapters on web services and databases (`pythonlearn.com`), and the Python `logging` HOWTO in the official docs.

## What's next

**Phase 2 Capstone: `defi-dashboard`.** You take this pipeline, extend it with your own sources and questions, add the Plotly dashboard, and publish it as a standalone GitHub repo. Before that, work through `exercises.py` (17 checked exercises across Parts 1 and 2).

## Commit your work

```bash
cd ~/Desktop/python-for-blockchain-analytics
git add phase-2-python-for-data/week-10-apis-sql/
git commit -m "phase-2/week-10: APIs and SQL from Python (lesson x3, exercises, pipeline)"
git push
```

Check `git status` first: no `.env`, `*.db`, `raw/` or `.cache/` should be staged.